<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/12-shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

# SHAP for Random Forests: Utah Wildfire Cause Classification

We will use a random forest to predict whether a Utah wildfire was caused by **human activity** or by **natural causes**, then use **SHAP** to explain the model's behavior.

### Learning goals
1. Build a tree-based classifier on a real tabular dataset.
2. Compare built-in random forest feature importance with SHAP-based explanations.
3. Interpret SHAP both globally (across many observations) and locally (for one prediction).
4. Practice separating **model explanation** from **causal explanation**.

### Suggested run order
Run the notebook from top to bottom once. Then revisit the discussion prompts and re-run the plotting cells as you experiment.

## 1) Install (if needed)

If you are running this notebook in a fresh environment, uncomment the next cell and run it once.

In [ ]:
# Uncomment and run if needed:
# %pip install -q shap scikit-learn pandas numpy matplotlib seaborn

## 2) Imports and setup

We import the usual data and plotting tools, plus:
- `KNNImputer` to fill in missing values
- `RandomForestClassifier` for the prediction model
- `shap` for model explanations

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from IPython.display import display

from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
shap.initjs()

## 3) Load the wildfire dataset

The original notebook read the file directly from GitHub. To make this version easier to run in different environments, we first look for a local copy and fall back to the URL if needed.

The target variable is `NWCG_CAUSE_CLASSIFICATION`:
- `Human`
- `Natural`

We also take `log(1 + FIRE_SIZE)` because fire size is highly skewed.

In [ ]:
DATA_URL = "https://github.com/esnt/Data/raw/main/Fires/utah_fires.csv"

df = pd.read_csv(DATA_URL)
df["FIRE_SIZE"] = np.log1p(df["FIRE_SIZE"])

# The full dataset is fairly large for repeated SHAP plotting in class,
# so we take a reproducible sample to keep runtime manageable.
df = df.sample(5000, random_state=RANDOM_STATE).reset_index(drop=True)

print("Data shape:", df.shape)
display(df.head())

### Quick inspection

Before fitting a model, it is worth checking:
- class balance
- feature names
- whether the variables look mostly numeric

In [ ]:
print("Class counts:")
display(df["NWCG_CAUSE_CLASSIFICATION"].value_counts())

print("\nFeature summary:")
display(df.describe(include="all").T)

## 4) Define features and split the data

We convert the target into a binary variable:
- `1` = Human-caused fire
- `0` = Natural fire

We also use a **stratified** train/test split so the class proportions stay similar in both sets.

In [ ]:
target_col = "NWCG_CAUSE_CLASSIFICATION"

X = df.drop(columns=target_col)
y = (df[target_col] == "Human").astype(int)
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Positive class rate (train):", y_train.mean().round(3))
print("Positive class rate (test):", y_test.mean().round(3))

## 5) Impute missing values

Random forests in scikit-learn do not handle missing values directly, so we impute them first.

We convert the imputed arrays back into pandas DataFrames so the feature names stay attached. That makes the SHAP visualizations much easier to read.

In [ ]:
imputer = KNNImputer(n_neighbors=5)

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names,
    index=X_train.index,
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index,
)

display(X_train_imputed.head())

## 6) Fit a random forest and evaluate it

This gives us a model worth explaining. SHAP is most useful when paired with a model that has learned something meaningful.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train_imputed, y_train)

y_pred = model.predict(X_test_imputed)
y_proba = model.predict_proba(X_test_imputed)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("F1 score:", round(f1_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=["Natural", "Human"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Natural", "Human"],
    cmap="Blues",
)
plt.title("Random Forest Confusion Matrix")
plt.show()

## 7) Built-in random forest feature importance

This is a useful baseline, but it is still a **global** summary. It does not tell us how a feature influenced any specific prediction.

In [ ]:
rf_importance = (
    pd.Series(model.feature_importances_, index=feature_names)
    .sort_values(ascending=True)
)

plt.figure(figsize=(8, 5))
rf_importance.tail(12).plot.barh(color="steelblue")
plt.title("Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Discussion prompt 2

- Which features does the random forest consider most important?
- What information is still missing if we only look at this plot?
- Could a feature be important for only a subset of fires, rather than for all fires equally?

## 8) Compute SHAP values

For tree models, `TreeExplainer` is usually the best starting point.

Important interpretation note:
- Positive SHAP values push a prediction toward the **Human** class.
- Negative SHAP values push a prediction toward the **Natural** class.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test_imputed)

positive_class_index = int(np.where(model.classes_ == 1)[0][0])

human_exp = shap.Explanation(
    values=shap_values.values[:, :, positive_class_index],
    base_values=shap_values.base_values[:, positive_class_index],
    data=X_test_imputed,
    feature_names=feature_names,
)

print("SHAP value tensor shape:", shap_values.values.shape)
print("Explanation matrix shape for the Human class:", human_exp.values.shape)

## 9) Global SHAP views

These plots answer slightly different questions:
- **Beeswarm plot**: which features matter most overall, and in what direction?
- **Bar plot**: which features have the largest average absolute effect?

In [ ]:
shap.plots.beeswarm(human_exp, max_display=12)

In [ ]:
shap.plots.bar(human_exp, max_display=12)

## Discussion prompt 3

- Compare the SHAP rankings with the random forest feature-importance chart. Where do they agree? Where do they differ?
- In the beeswarm plot, do high feature values always push predictions in the same direction?
- Which features seem to have the most variable effect across observations?

## 10) Local explanation for one fire

Global summaries are helpful, but SHAP is especially valuable when we zoom in on a single observation.

The next cells inspect one specific fire from the test set.

In [ ]:
example_idx = 11

print("Observed class:", "Human" if y_test.iloc[example_idx] == 1 else "Natural")
print("Predicted probability of Human cause:", round(y_proba[example_idx], 3))
display(X_test_imputed.iloc[[example_idx]])

In [ ]:
shap.plots.bar(human_exp[example_idx], show_data=True)

In [ ]:
shap.plots.waterfall(human_exp[example_idx], max_display=12)

In [ ]:
shap.plots.force(human_exp[example_idx])

## Discussion prompt 4

- Which features pushed this particular prediction toward `Human`?
- Which features pushed back toward `Natural`?
- If you had only the global importance plot, what would you miss about this one example?

## 11) Dependence-style plot

This plot focuses on one feature at a time. Here we color by `DISCOVERY_DOY` so we can see whether the effect of latitude may interact with time of year.

In [ ]:
shap.dependence_plot("LATITUDE", human_exp.values, X_test_imputed, interaction_index="DISCOVERY_DOY")

## 12) Interpreting carefully

SHAP explains the **model's prediction rule**, not the real-world data-generating process.

That means:
- A large SHAP value does **not** prove a feature causes fires.
- Correlated predictors can share or split attribution in non-obvious ways.
- If the model is weak or biased, the SHAP explanations can also be misleading.

## Final student discussion questions

1. Why is SHAP usually described as a **local** explanation method, even though we can build global summaries from it?
2. What is the difference between a feature being important on average and a feature being decisive for one individual prediction?
3. Which plot in this notebook was easiest for you to interpret? Which was hardest? Why?
4. If two predictors are strongly correlated, how might that complicate the SHAP interpretation?
5. What would change if we trained a different model class, such as logistic regression or gradient boosting, on the same data?
6. Why should we avoid turning a SHAP explanation into a causal claim?